In [1]:
# Import libraries

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [2]:
# Load raw transaction dataset

df = pd.read_csv("../data/raw/transactions.csv")

# Display first 5 rows
df.head()

,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,ip_risk_score,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,ATM,278.19,278.19,4.25,...,0.123,standard,263,0.522,0,0.223,0,0,0.0,0
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,4.24,...,0.569,standard,947,0.475,0,0.268,0,1,0.0,0
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,2.70,...,0.437,enhanced,367,0.939,0,0.176,0,0,0.0,0
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,2.22,...,0.594,standard,147,0.551,0,0.391,0,0,0.0,0
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,3.61,...,0.121,enhanced,257,0.894,0,0.257,0,0,0.0,0


In [3]:
# Check original dataset shape

print("Original dataset shape:", df.shape)

Original dataset shape: (11400, 26)


In [4]:
# Check missing values before cleaning

df.isnull().sum()

transaction_id                 0
customer_id                    0
timestamp                     29
home_country                   0
source_currency                0
dest_currency                  0
channel                        0
amount_src                     0
amount_usd                   305
fee                          295
exchange_rate_src_to_dest      0
device_id                      0
new_device                     0
ip_address                   305
ip_country                   301
location_mismatch              0
ip_risk_score                  0
kyc_tier                     300
account_age_days               0
device_trust_score           295
chargeback_history_count       0
risk_score_internal            0
txn_velocity_1h                0
txn_velocity_24h               0
corridor_risk                  0
is_fraud                       0
dtype: int64

In [5]:
# Check duplicate rows before cleaning

print("Duplicate rows before cleaning:", df.duplicated().sum())

Duplicate rows before cleaning: 200


In [6]:
# Remove exact duplicate records

df = df.drop_duplicates()

# Confirm duplicates removed
print("Duplicate rows after cleaning:", df.duplicated().sum())

Duplicate rows after cleaning: 0


In [7]:
# Convert timestamp column to datetime format
# Invalid timestamps will be converted to NaT

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

In [8]:
# Convert amount_src to numeric
# Invalid values will be converted to NaN

df["amount_src"] = pd.to_numeric(
    df["amount_src"],
    errors="coerce"
)

In [9]:
# Standardize text columns by converting to lowercase and removing extra spaces

text_cols = [
    "home_country",
    "source_currency",
    "dest_currency",
    "channel",
    "ip_country",
    "kyc_tier"
]

for col in text_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.lower()
        .str.strip()
    )

In [10]:
# Fix spelling inconsistencies in categorical columns

df["channel"] = df["channel"].replace({
    "mobille": "mobile",
    "weeb": "web"
})

df["kyc_tier"] = df["kyc_tier"].replace({
    "enhancd": "enhanced",
    "standrd": "standard",
    "nan": "unknown"
})

df["ip_country"] = df["ip_country"].replace({
    "nan": "unknown"
})

In [11]:
# Check cleaned category values

print("Channel values:")
print(df["channel"].unique())

print("\nKYC tier values:")
print(df["kyc_tier"].unique())

print("\nIP country values:")
print(df["ip_country"].unique())

Channel values:
<ArrowStringArray>
['atm', 'web', 'mobile', 'unknown']
Length: 4, dtype: str

KYC tier values:
<ArrowStringArray>
['standard', 'enhanced', 'low', nan, 'unknown']
Length: 5, dtype: str

IP country values:
<ArrowStringArray>
['us', 'ca', nan, 'uk', 'unknown']
Length: 5, dtype: str


In [12]:
# Fill missing numerical values using median

numeric_cols = [
    "amount_src",
    "amount_usd",
    "fee",
    "device_trust_score"
]

for col in numeric_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

In [13]:
# Fill missing categorical values with 'unknown'

categorical_cols = [
    "ip_country",
    "kyc_tier"
]

for col in categorical_cols:
    df[col] = df[col].fillna("unknown")

In [14]:
# Remove rows with invalid timestamps

df = df.dropna(
    subset=["timestamp"]
)

In [15]:
# Remove invalid numerical values

df = df[df["fee"] >= 0]
df = df[df["txn_velocity_1h"] >= 0]
df = df[df["device_trust_score"] >= 0]

# Keep IP risk score between 0 and 1

df = df[df["ip_risk_score"].between(0, 1)]

In [16]:
# Create temporal features from timestamp

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["month"] = df["timestamp"].dt.month
df["year"] = df["timestamp"].dt.year

In [17]:
# Create destination amount column
# This estimates the destination currency amount using the source amount
# and the source-to-destination exchange rate

df["amount_dest"] = (
    df["amount_src"] *
    df["exchange_rate_src_to_dest"]
).round(2)

In [18]:
# Drop identifier columns that do not help model prediction

identifier_cols = [
    "transaction_id",
    "customer_id",
    "device_id",
    "ip_address"
]

df = df.drop(
    columns=identifier_cols
)

In [19]:
# Convert binary columns from boolean to integer

binary_cols = [
    "new_device",
    "location_mismatch",
    "is_fraud"
]

for col in binary_cols:
    df[col] = df[col].astype(int)

In [20]:
# Convert categorical columns to category data type

cat_cols = [
    "home_country",
    "source_currency",
    "dest_currency",
    "channel",
    "ip_country",
    "kyc_tier",
    "day_of_week"
]

for col in cat_cols:
    df[col] = df[col].astype("category")

In [21]:
# Reset index after cleaning

df = df.reset_index(drop=True)

In [22]:
# Validate cleaned dataset structure

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10940 entries, 0 to 10939
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   timestamp                  10940 non-null  datetime64[us, UTC]
 1   home_country               10940 non-null  category           
 2   source_currency            10940 non-null  category           
 3   dest_currency              10940 non-null  category           
 4   channel                    10940 non-null  category           
 5   amount_src                 10940 non-null  float64            
 6   amount_usd                 10940 non-null  float64            
 7   fee                        10940 non-null  float64            
 8   exchange_rate_src_to_dest  10940 non-null  float64            
 9   new_device                 10940 non-null  int64              
 10  ip_country                 10940 non-null  category           
 11  location_mism

In [23]:
# Confirm no missing values remain

df.isnull().sum()

timestamp                    0
home_country                 0
source_currency              0
dest_currency                0
channel                      0
amount_src                   0
amount_usd                   0
fee                          0
exchange_rate_src_to_dest    0
new_device                   0
ip_country                   0
location_mismatch            0
ip_risk_score                0
kyc_tier                     0
account_age_days             0
device_trust_score           0
chargeback_history_count     0
risk_score_internal          0
txn_velocity_1h              0
txn_velocity_24h             0
corridor_risk                0
is_fraud                     0
hour                         0
day_of_week                  0
month                        0
year                         0
amount_dest                  0
dtype: int64

In [24]:
# Confirm no duplicate rows remain

print("Final duplicate rows:", df.duplicated().sum())

Final duplicate rows: 0


In [25]:
# Check final dataset shape

print("Cleaned dataset shape:", df.shape)

Cleaned dataset shape: (10940, 27)


In [26]:
# Check final fraud distribution

print("Fraud class distribution:")
print(df["is_fraud"].value_counts())

fraud_rate = df["is_fraud"].mean() * 100
print(f"\nFraud Rate: {fraud_rate:.2f}%")

Fraud class distribution:
is_fraud
0    9951
1     989
Name: count, dtype: int64

Fraud Rate: 9.04%


In [27]:
# Display cleaned dataset

df.head()

,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,exchange_rate_src_to_dest,new_device,...,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud,hour,day_of_week,month,year,amount_dest
0,2022-10-03 18:40:59.468549+00:00,us,usd,cad,atm,278.19,278.19,4.25,1.351351,0,...,0.223,0,0,0.0,0,18,Monday,10,2022,375.93
1,2022-10-03 20:39:38.468549+00:00,ca,cad,mxn,web,208.51,154.29,4.24,12.758621,1,...,0.268,0,1,0.0,0,20,Monday,10,2022,2660.30
2,2022-10-03 23:02:43.468549+00:00,us,usd,cny,mobile,160.33,160.33,2.70,7.142857,0,...,0.176,0,0,0.0,0,23,Monday,10,2022,1145.21
3,2022-10-04 01:08:53.468549+00:00,us,usd,eur,mobile,59.41,59.41,2.22,0.925926,0,...,0.391,0,0,0.0,0,1,Tuesday,10,2022,55.01
4,2022-10-04 09:35:03.468549+00:00,us,usd,inr,mobile,200.96,200.96,3.61,83.333333,0,...,0.257,0,0,0.0,0,9,Tuesday,10,2022,16746.67


In [28]:
# Save cleaned dataset to processed folder

df.to_csv(
    "../data/processed/cleaned_transactions.csv",
    index=False
)

In [29]:
# Confirm file saved successfully

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
